In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

OPENALEX_CSV = Path("openalex_works.csv")                 # <- set your filename
S2_CSV       = Path("semantic_scholar_results.csv")         # <- your existing file
OUT_CSV      = Path("stageA_combined_oa_s2.csv")


In [ ]:
df_oa = pd.read_csv(OPENALEX_CSV)
df_s2 = pd.read_csv(S2_CSV)

print("OpenAlex:", df_oa.shape, "| cols:", list(df_oa.columns))
print("S2      :", df_s2.shape, "| cols:", list(df_s2.columns))

df_oa.head(1)

In [ ]:
df_s2.head(1)

In [ ]:
def standardize_openalex(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["source"] = "openalex"
    d["source_id"] = d.get("openalex_id")
    d["citation_count"] = d.get("cited_by")
    # keep your columns; make sure "abstract" exists
    for col in ["abstract", "doi", "title", "year", "venue", "type", "authors(first6)", "query"]:
        if col not in d.columns:
            d[col] = np.nan
    return d[[
        "source","source_id","query","title","year","venue","type",
        "authors(first6)","doi","citation_count","abstract","openalex_id"
    ]]

def standardize_s2(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["source"] = "semantic_scholar"
    d["source_id"] = d.get("paperId")
    d["citation_count"] = d.get("citationCount")
    for col in ["abstract", "doi", "title", "year", "venue", "authors(first6)", "query", "paperId", "s2_url"]:
        if col not in d.columns:
            d[col] = np.nan
    d["type"] = np.nan  # S2 search doesn't consistently return type
    d["openalex_id"] = np.nan
    return d[[
        "source","source_id","query","title","year","venue","type",
        "authors(first6)","doi","citation_count","abstract","paperId","s2_url"
    ]]

oa_std = standardize_openalex(df_oa)
s2_std = standardize_s2(df_s2)

combined_raw = pd.concat([oa_std, s2_std], ignore_index=True)
print("Combined raw:", combined_raw.shape)
combined_raw.head(5)


In [ ]:
def normalize_doi(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x)
    x = re.sub(r"^doi:\s*", "", x)
    x = x.strip()
    return x or None

def normalize_title(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    # remove punctuation-ish, collapse spaces
    x = re.sub(r"[\u2010-\u2015]", "-", x)   # normalize dash variants
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x or None


In [ ]:
stageA = combined_raw.copy()

stageA["doi_norm"] = stageA["doi"].map(normalize_doi)
stageA["title_norm"] = stageA["title"].map(normalize_title)

# numeric citations
stageA["citation_count"] = pd.to_numeric(stageA["citation_count"], errors="coerce")

# keep only rows with a title (hard to use without)
stageA = stageA[stageA["title_norm"].notna()].reset_index(drop=True)

print("StageA rows (after title filter):", len(stageA))
stageA.head(5)


In [ ]:
def longest_text(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    return max(vals, key=len)

def most_common_or_longest(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    vc = pd.Series(vals).value_counts()
    if len(vc) and vc.iloc[0] >= 2:
        return vc.index[0]
    return max(vals, key=len)

def first_nonnull(series):
    for v in series:
        if pd.notna(v) and v not in ("", None):
            return v
    return None

# Within-source grouping key: prefer DOI, else source_id, else title+year
def within_source_key(df):
    k = []
    for _, r in df.iterrows():
        if r["doi_norm"]:
            k.append(f"doi:{r['doi_norm']}")
        elif pd.notna(r["source_id"]):
            k.append(f"id:{r['source']}:{r['source_id']}")
        else:
            y = int(r["year"]) if pd.notna(r["year"]) else ""
            k.append(f"ty:{r['title_norm']}|{y}")
    return pd.Series(k, index=df.index)

stageA["within_key"] = within_source_key(stageA)

agg_map = {
    "source": lambda s: s.iloc[0],
    "source_id": first_nonnull,
    "title": most_common_or_longest,
    "title_norm": lambda s: s.iloc[0],
    "year": lambda s: pd.to_numeric(s, errors="coerce").dropna().median() if s.notna().any() else np.nan,
    "venue": most_common_or_longest,
    "type": most_common_or_longest,
    "authors(first6)": most_common_or_longest,
    "doi": first_nonnull,
    "doi_norm": first_nonnull,
    "citation_count": lambda s: pd.to_numeric(s, errors="coerce").max(),
    "abstract": longest_text,
    "query": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()])))
    ,
    "openalex_id": first_nonnull,
    "paperId": first_nonnull,
    "s2_url": first_nonnull,
}

stageA_dedup = (
    stageA
    .groupby(["source", "within_key"], as_index=False)
    .agg(agg_map)
    .drop(columns=["within_key"])
)

print("After within-source dedupe:", stageA_dedup.shape)
stageA_dedup.head(5)


In [ ]:
def cross_source_merge_key(r):
    if isinstance(r["doi_norm"], str) and r["doi_norm"]:
        return f"doi:{r['doi_norm']}"
    if pd.notna(r["year"]):
        return f"ty:{r['title_norm']}|{int(round(float(r['year'])))}"
    return f"t:{r['title_norm']}"

stageA_dedup["merge_key"] = stageA_dedup.apply(cross_source_merge_key, axis=1)

def merge_sources(group: pd.DataFrame) -> dict:
    # provenance
    sources = sorted(set(group["source"].dropna().tolist()))
    source_ids = {src: group.loc[group["source"] == src, "source_id"].dropna().astype(str).unique().tolist()
                  for src in sources}

    out = {
        "merge_key": group["merge_key"].iloc[0],
        "sources": "; ".join(sources),
        "source_count": len(sources),
        "source_ids": str(source_ids),
        "title": most_common_or_longest(group["title"]),
        "year": pd.to_numeric(group["year"], errors="coerce").dropna().median() if group["year"].notna().any() else np.nan,
        "venue": most_common_or_longest(group["venue"]),
        "type": most_common_or_longest(group["type"]),
        "authors(first6)": most_common_or_longest(group["authors(first6)"]),
        "doi": first_nonnull(group["doi"]),
        "doi_norm": first_nonnull(group["doi_norm"]),
        "citation_count_max": pd.to_numeric(group["citation_count"], errors="coerce").max(),
        "abstract": longest_text(group["abstract"]),
        "queries": "; ".join(sorted(set(
            q for q in group["query"].dropna().tolist()
            if isinstance(q, str) and q.strip()
        ))),
        # keep ids/urls if present
        "openalex_id": first_nonnull(group.get("openalex_id", pd.Series([], dtype=object))),
        "paperId": first_nonnull(group.get("paperId", pd.Series([], dtype=object))),
        "s2_url": first_nonnull(group.get("s2_url", pd.Series([], dtype=object))),
    }
    return out

merged_records = [merge_sources(g) for _, g in stageA_dedup.groupby("merge_key")]
df_stageA = pd.DataFrame(merged_records)

# Optional: sort by citation_count_max (just for inspection; ranking comes later)
df_stageA = df_stageA.sort_values(by="citation_count_max", ascending=False, na_position="last").reset_index(drop=True)

print("Stage A merged unique records:", df_stageA.shape)
df_stageA.head(25)


In [ ]:
# How many are DOI-merged vs title-merged?
df_stageA["merge_kind"] = df_stageA["merge_key"].str.split(":", n=1).str[0]
df_stageA["has_abstract"] = df_stageA["abstract"].notna() & (df_stageA["abstract"].str.len() > 50)

print(df_stageA["merge_kind"].value_counts(dropna=False))
print("\nSources coverage:")
print(df_stageA["sources"].value_counts().head(10))

print("\nAbstract coverage:", df_stageA["has_abstract"].mean(), "(fraction with decent abstract)")

# Show examples where both sources matched
both = df_stageA[df_stageA["source_count"] >= 2].copy()
print("\nRecords with BOTH OpenAlex + S2:", len(both))
both.head(20)


In [ ]:
df_stageA.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV.resolve())

In [ ]:
# Use the within-source-deduped table from Cell 6: stageA_dedup
# (If you didn't keep it, reload from your earlier notebook state or rebuild quickly.)

def pct(x): 
    return round(100 * x, 2)

summary = (
    stageA_dedup.assign(
        has_doi = stageA_dedup["doi_norm"].notna(),
        has_year = stageA_dedup["year"].notna(),
        has_abs = stageA_dedup["abstract"].notna() & (stageA_dedup["abstract"].astype(str).str.len() > 50),
    )
    .groupby("source")[["has_doi","has_year","has_abs"]]
    .mean()
)

summary = summary.applymap(pct)
summary


In [ ]:
oa = stageA_dedup[stageA_dedup["source"]=="openalex"].copy()
s2 = stageA_dedup[stageA_dedup["source"]=="semantic_scholar"].copy()

oa_dois = set(oa["doi_norm"].dropna().astype(str))
s2_dois = set(s2["doi_norm"].dropna().astype(str))

print("OpenAlex doi_norm count:", len(oa_dois))
print("S2 doi_norm count      :", len(s2_dois))
print("DOI intersection       :", len(oa_dois & s2_dois))

# Show a few intersecting DOIs if any
list(sorted(list(oa_dois & s2_dois)))[:20]


In [ ]:
def ty_key(df):
    d = df.copy()
    d["year_i"] = pd.to_numeric(d["year"], errors="coerce").round().astype("Int64")
    d["ty"] = np.where(d["year_i"].notna(), d["title_norm"] + "|" + d["year_i"].astype(str), None)
    return set(d["ty"].dropna().astype(str))

oa_ty = ty_key(oa)
s2_ty = ty_key(s2)

print("OpenAlex title|year keys:", len(oa_ty))
print("S2 title|year keys      :", len(s2_ty))
print("Intersection (title|yr) :", len(oa_ty & s2_ty))

list(sorted(list(oa_ty & s2_ty)))[:10]


In [ ]:
oa = stageA_dedup[stageA_dedup["source"]=="openalex"].copy()
s2 = stageA_dedup[stageA_dedup["source"]=="semantic_scholar"].copy()

oa_titles = set(oa["title_norm"].dropna().astype(str))
s2_titles = set(s2["title_norm"].dropna().astype(str))

print("OpenAlex title_norm:", len(oa_titles))
print("S2 title_norm      :", len(s2_titles))
print("Title intersection :", len(oa_titles & s2_titles))

# show a few exact-title matches (if any)
list(sorted(list(oa_titles & s2_titles)))[:20]


In [ ]:
def contains_any(text, keywords):
    if not isinstance(text, str):
        return False
    t = text.lower()
    return any(k in t for k in keywords)

keywords = [
    "retail", "brick-and-mortar", "brick and mortar", "in-store", "in store",
    "offline", "store", "supermarket", "grocery", "point of sale", "pos"
]

tmp = stageA_dedup.copy()
tmp["text"] = (tmp["title"].fillna("") + " " + tmp["abstract"].fillna("")).astype(str)

tmp["has_retail_terms"] = tmp["text"].apply(lambda x: contains_any(x, keywords))

tmp.groupby("source")["has_retail_terms"].mean().mul(100).round(2)


---

# Stage B

---

In [ ]:
import pandas as pd
from pathlib import Path

STAGEA_CSV = Path("stageA_combined_oa_s2.csv")  # <- your saved Stage A output
df_stageA = pd.read_csv(STAGEA_CSV)

# Keep only what we need for later ranking (title+abstract + a few metadata/tie-breakers)
cols_keep = [
    "merge_key","sources","title","year","venue","type","doi_norm",
    "citation_count_max","abstract","queries","openalex_id","paperId","s2_url"
]
df_stageA = df_stageA[[c for c in cols_keep if c in df_stageA.columns]].copy()

print("Loaded:", df_stageA.shape)
df_stageA.head(3)


In [ ]:
CHAPTER_ID = "2.1"
CHAPTER_TITLE = "Definition and concepts of Dynamic Pricing in brick-and-mortar retail"

CHAPTER_TEXT_DE = (
    "Die theoretischen Grundlagen des Dynamic Pricing werden detailliert beschrieben. "
    "Es wird erläutert, wie Dynamic Pricing im Einzelhandel funktioniert und welche "
    "unterschiedlichen Ansätze es gibt, z. B. zeitabhängige Preise oder an die Nachfrage "
    "gekoppelte Preisänderungen. Es wird nicht auf operative Details der Preisgestaltung "
    "oder spezifische Implementierungsstrategien eingegangen. Der Online-Handel wird "
    "explizit ausgeklammert."
)

# Minimal English scope (no exclusions enforced yet; just captured for later)
CHAPTER_SCOPE_EN = (
    "Explain the theoretical foundations, definitions, and core concepts of dynamic pricing "
    "in brick-and-mortar (offline / in-store) retail. Cover approaches such as time-dependent "
    "pricing and demand-linked price changes. Do not focus on operational implementation details."
)

CHAPTER_EXCLUSIONS_FOR_LATER = [
    "online retail", "e-commerce", "webshop", "internet retail", "digital-only retail"
]

chapter_spec = {
    "chapter_id": CHAPTER_ID,
    "title": CHAPTER_TITLE,
    "scope_en": CHAPTER_SCOPE_EN,
    "original_text_de": CHAPTER_TEXT_DE,
    "exclusions_for_later": CHAPTER_EXCLUSIONS_FOR_LATER,
}

chapter_spec


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class ChapterQueryProfile(BaseModel):
    chapter_id: str = Field(..., description="Chapter identifier (e.g., '2.1').")
    language: str = Field("en", description="Language of the queries.")
    
    main_query: str = Field(..., description="One broad query string capturing the chapter scope.")
    facet_queries: List[str] = Field(..., description="6–10 narrower queries covering key sub-aspects.")
    
    keywords: List[str] = Field(..., description="15–30 keywords/phrases to help downstream matching.")
    key_concepts: List[str] = Field(..., description="8–15 conceptual phrases (more abstract than keywords).")
    
    exclusions_for_later: List[str] = Field(..., description="Terms to exclude later (NOT applied yet).")
    notes: Optional[str] = Field(None, description="Short notes about intent/coverage limits.")

print("Schema ready.")


In [ ]:
import os
from agents import Agent, Runner, ModelSettings

# Ensure OPENAI_API_KEY is set in your environment before running.
# Example: export OPENAI_API_KEY="..."
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."

query_profile_agent = Agent(
    name="Chapter Query Profiler",
    model="gpt-5-nano",
    model_settings=ModelSettings(
        top_p=1.0,
        verbosity="low",
    ),
    instructions=(
        "You create search query profiles for academic literature retrieval.\n"
        "Return ONLY the structured output fields (no extra text).\n\n"
        "Constraints:\n"
        "- language must be 'en'\n"
        "- main_query: <= 18 words, plain text, allow quotes for phrases\n"
        "- facet_queries: 6–10 items, each <= 14 words, plain text, allow quotes\n"
        "- keywords: 15–30 items (mix of single terms + short phrases)\n"
        "- key_concepts: 8–15 items (conceptual phrases; e.g., 'intertemporal price discrimination')\n"
        "- exclusions_for_later: copy from input + add up to 5 close variants\n"
        "- Do NOT apply exclusions; just list them.\n"
        "- Do NOT include implementation/algorithm engineering terms unless central to concepts.\n"
        "- Focus on brick-and-mortar/offline/in-store retail context.\n"
    ),
    output_type=ChapterQueryProfile,
)

print("Agent ready.")


In [ ]:
import json
from pathlib import Path
from agents import Runner

payload = {
    "chapter_id": chapter_spec["chapter_id"],
    "title": chapter_spec["title"],
    "scope_en": chapter_spec["scope_en"],
    "exclusions_for_later": chapter_spec["exclusions_for_later"],
    "original_text_de": chapter_spec["original_text_de"],
}

prompt = (
    "Create a ChapterQueryProfile for academic literature retrieval.\n"
    "Return ONLY the structured output fields required by the schema.\n\n"
    "CHAPTER_SPEC_JSON:\n"
    f"{json.dumps(payload, ensure_ascii=False, indent=2)}"
)

# Jupyter: use `await` (NOT run_sync, NOT asyncio.run)
result = await Runner.run(query_profile_agent, prompt)

profile = result.final_output
profile_dict = profile.model_dump()

Path("stageB_query_profile_2_1.json").write_text(
    json.dumps(profile_dict, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


In [ ]:
import pandas as pd

df_facets = pd.DataFrame({"facet_queries": profile_dict["facet_queries"]})
df_keywords = pd.DataFrame({"keywords": profile_dict["keywords"]})
df_concepts = pd.DataFrame({"key_concepts": profile_dict["key_concepts"]})
df_excl = pd.DataFrame({"exclusions_for_later": profile_dict["exclusions_for_later"]})

print("Main query:\n", profile_dict["main_query"])
display(df_facets.head(20))
display(df_keywords.head(40))
display(df_concepts.head(40))
display(df_excl.head(40))


---

# Stage C

---

## C.1

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

STAGEA_CSV = Path("stageA_combined_oa_s2.csv")
PROFILE_JSON = Path("stageB_query_profile_2_1.json")

df = pd.read_csv(STAGEA_CSV)
profile = json.loads(PROFILE_JSON.read_text(encoding="utf-8"))

print("StageA:", df.shape)
print("Profile keys:", list(profile.keys()))
df.head(1)


StageA: (2039, 19)
Profile keys: ['chapter_id', 'language', 'main_query', 'facet_queries', 'keywords', 'key_concepts', 'exclusions_for_later', 'notes']


,merge_key,sources,source_count,source_ids,title,year,venue,type,authors(first6),doi,doi_norm,citation_count_max,abstract,queries,openalex_id,paperId,s2_url,merge_kind,has_abstract
0,doi:10.1007/s42979-021-00592-x,openalex,1,{'openalex': ['https://openalex.org/W313502870...,"Machine Learning: Algorithms, Real-World Appli...",2021.0,SN Computer Science,review,Iqbal H. Sarker,https://doi.org/10.1007/s42979-021-00592-x,10.1007/s42979-021-00592-x,4635,NaN,time-based pricing time-dependent pricing dyna...,https://openalex.org/W3135028703,NaN,NaN,doi,False


In [2]:
def build_query_text(profile: dict) -> str:
    parts = []
    # Give the main query more weight by repeating it
    main = profile["main_query"]
    parts.extend([main, main])

    # Facets: medium weight
    for fq in profile.get("facet_queries", []):
        parts.append(fq)

    # Keywords + concepts: lighter weight (but still useful)
    parts.extend(profile.get("keywords", []))
    parts.extend(profile.get("key_concepts", []))

    return " ".join(parts)

chapter_query_text = build_query_text(profile)
print(chapter_query_text[:600] + " ...")


Dynamic pricing concepts for brick-and-mortar retail: time-dependent and demand-linked strategies Dynamic pricing concepts for brick-and-mortar retail: time-dependent and demand-linked strategies definitions and terminology of dynamic pricing in offline retail time-based pricing approaches in-store demand-linked price adjustments and elasticity competitor-agnostic vs competitor-aware pricing in physical stores customer perceived fairness and pricing transparency offline data sources for in-store pricing decisions dynamic pricing offline retail brick-and-mortar in-store pricing price discrimina ...


In [3]:
def safe_str(x):
    return "" if pd.isna(x) else str(x)

df = df.copy()
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")
df["doc_text"] = (df["title"].astype(str) + "\n\n" + df["abstract"].astype(str)).str.strip()

# Basic stats
df["has_abstract"] = df["abstract"].str.len() > 50
print("Docs:", len(df))
print("Abstract coverage:", df["has_abstract"].mean())
df[["title","has_abstract","citation_count_max"]].head(5)


Docs: 2039
Abstract coverage: 0.7788131436978911


,title,has_abstract,citation_count_max
0,"Machine Learning: Algorithms, Real-World Appli...",False,4635
1,The Top 10 fungal pathogens in molecular plant...,True,4404
2,Artificial Intelligence (AI): Multidisciplinar...,False,3619
3,Digital Business Strategy: Toward a Next Gener...,True,3569
4,World agriculture towards 2030/2050: the 2012 ...,True,3140


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Keep it efficient: cap vocab size; include bigrams for phrases like "brick and mortar"
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=200_000,
    min_df=2
)

X = vectorizer.fit_transform(df["doc_text"])
q = vectorizer.transform([chapter_query_text])

tfidf_score = cosine_similarity(q, X).ravel()
df["score_tfidf"] = tfidf_score

df[["title","score_tfidf"]].sort_values("score_tfidf", ascending=False).head(10)


,title,score_tfidf
2031,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.310356
1491,Leveraging Microservices Architecture for Dyna...,0.299554
1765,When should grocery stores adopt time‐based pr...,0.280171
2009,The Role of Dynamic Pricing to Improve Revenue...,0.274905
567,Price Discrimination in E-Commerce? An Examina...,0.271994
1348,Optimal Rebate Strategies Under Dynamic Pricing,0.261373
1896,OPTIMASI DYNAMIC PRICING MENGGUNAKANMETODE ALG...,0.257710
1349,Exploring customers’ likeliness to use e-servi...,0.254061
1921,COMPARING APPROACHES: A SCIENTIFIC OVERVIEW OF...,0.246024
1952,Machine Learning for Retail Pricing Optimization,0.237998


In [5]:
# If missing, treat as 0
cites = pd.to_numeric(df.get("citation_count_max", 0), errors="coerce").fillna(0).clip(lower=0)

# log1p smooths extremes: 0->0, 10->~2.4, 100->~4.6, 1000->~6.9
df["score_cite"] = np.log1p(cites)

# Normalize cite score to 0..1 for stable weighting
if df["score_cite"].max() > 0:
    df["score_cite_norm"] = df["score_cite"] / df["score_cite"].max()
else:
    df["score_cite_norm"] = 0.0

# Final baseline score: mostly relevance, tiny citation bump
CITE_WEIGHT = 0.08
df["score_stageC1"] = df["score_tfidf"] * (1 - CITE_WEIGHT) + df["score_cite_norm"] * CITE_WEIGHT

df[["title","score_tfidf","score_cite_norm","score_stageC1"]].sort_values("score_stageC1", ascending=False).head(10)


,title,score_tfidf,score_cite_norm,score_stageC1
567,Price Discrimination in E-Commerce? An Examina...,0.271994,0.628234,0.300494
1491,Leveraging Microservices Architecture for Dyna...,0.299554,0.284057,0.298314
2031,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.310356,0.000000,0.285527
1348,Optimal Rebate Strategies Under Dynamic Pricing,0.261373,0.385957,0.271340
1765,When should grocery stores adopt time‐based pr...,0.280171,0.130143,0.268169
1349,Exploring customers’ likeliness to use e-servi...,0.254061,0.381311,0.264241
980,Personalized pricing and price fairness,0.230030,0.534360,0.254376
2009,The Role of Dynamic Pricing to Improve Revenue...,0.274905,0.000000,0.252913
1896,OPTIMASI DYNAMIC PRICING MENGGUNAKANMETODE ALG...,0.257710,0.000000,0.237093
1552,Real-time dynamic Pricing for multiproduct mod...,0.233429,0.230514,0.233196


In [6]:
TOP_N = 50

cols_show = [
    "score_stageC1","score_tfidf","score_cite_norm",
    "title","year","venue","sources","doi_norm","citation_count_max","has_abstract"
]

top = df.sort_values("score_stageC1", ascending=False).head(TOP_N)[cols_show].reset_index(drop=True)
top


,score_stageC1,score_tfidf,score_cite_norm,title,year,venue,sources,doi_norm,citation_count_max,has_abstract
0,0.300494,0.271994,0.628234,Price Discrimination in E-Commerce? An Examina...,2011.0,MIS Q.,semantic_scholar,10.2307/23043490,200,False
1,0.298314,0.299554,0.284057,Leveraging Microservices Architecture for Dyna...,2024.0,arXiv.org,semantic_scholar,10.48550/arxiv.2411.01636,10,True
2,0.285527,0.310356,0.000000,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,2025.0,EPRA International Journal of Economic and Bus...,semantic_scholar,10.36713/epra21984,0,True
3,0.271340,0.261373,0.385957,Optimal Rebate Strategies Under Dynamic Pricing,2017.0,Operational Research,semantic_scholar,10.1287/opre.2017.1642,25,False
4,0.268169,0.280171,0.130143,When should grocery stores adopt time‐based pr...,2023.0,Production and operations management,semantic_scholar,10.1111/poms.14010,2,True
5,0.264241,0.254061,0.381311,Exploring customers’ likeliness to use e-servi...,2020.0,Electronic Markets,openalex,10.1007/s12525-020-00445-0,24,False
6,0.254376,0.230030,0.534360,Personalized pricing and price fairness,2015.0,International Journal of Industrial Organization,openalex,10.1016/j.ijindorg.2015.11.004,90,False
7,0.252913,0.274905,0.000000,The Role of Dynamic Pricing to Improve Revenue...,2025.0,Maneggio,semantic_scholar,10.62872/fas50936,0,True
8,0.237093,0.257710,0.000000,OPTIMASI DYNAMIC PRICING MENGGUNAKANMETODE ALG...,2013.0,NaN,semantic_scholar,NaN,0,False
9,0.233196,0.233429,0.230514,Real-time dynamic Pricing for multiproduct mod...,2009.0,American Control Conference,semantic_scholar,10.1109/acc.2009.5160689,6,False


## C.2

In [7]:
import numpy as np
import pandas as pd

# We assume you already have:
# - df with columns: merge_key, title, abstract, doc_text, score_tfidf, score_cite_norm
# - vectorizer, X already computed from Stage C.1 (TF-IDF matrix for df["doc_text"])
# If you don't, re-run Stage C.1 cells (vectorizer + X + score_tfidf).

from sklearn.metrics.pairwise import cosine_similarity

# Use main + facets as separate "retrieval queries"
query_texts = [profile["main_query"]] + profile.get("facet_queries", [])

TOP_PER_QUERY = 250  # tune: 150–400 are common; larger = more embedding cost

# Compute TF-IDF similarity per query and union top ids
pool_keys = set()
for qt in query_texts:
    qv = vectorizer.transform([qt])
    sims = cosine_similarity(qv, X).ravel()
    top_idx = np.argsort(-sims)[:TOP_PER_QUERY]
    pool_keys.update(df.iloc[top_idx]["merge_key"].astype(str).tolist())

df_pool = df[df["merge_key"].astype(str).isin(pool_keys)].copy()
df_pool = df_pool.drop_duplicates(subset=["merge_key"]).reset_index(drop=True)

print("Facet-union embedding pool size:", len(df_pool))
df_pool[["title","score_tfidf","score_stageC1"]].head(5)


Facet-union embedding pool size: 724


,title,score_tfidf,score_stageC1
0,"A Metaverse: Taxonomy, Components, Application...",0.027064,0.095171
1,Challenges of Big Data analysis,0.023072,0.089879
2,"The Rise of Supermarkets in Africa, Asia, and ...",0.032663,0.098167
3,The impact of customer satisfaction and relati...,0.025070,0.090756
4,Organizations and Markets,0.025967,0.091065


In [8]:
from pathlib import Path

MAX_CHARS_PER_DOC = 3500
BATCH_SIZE = 64
EMBED_MODEL = "text-embedding-3-small"

CACHE_DIR = Path(".embed_cache")
CACHE_DIR.mkdir(exist_ok=True)

POOL_TAG = f"facetUnion_q{len(query_texts)}_top{TOP_PER_QUERY}_n{len(df_pool)}"
DOC_EMBED_NPZ = CACHE_DIR / f"doc_embeds_{EMBED_MODEL}_{POOL_TAG}.npz"
QUERY_EMBED_JSON = CACHE_DIR / f"query_embeds_{EMBED_MODEL}_{POOL_TAG}.json"

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

df_pool["doc_text_trunc"] = (df_pool["title"].fillna("") + "\n\n" + df_pool["abstract"].fillna("")).apply(
    lambda x: truncate_text(x, MAX_CHARS_PER_DOC)
)

print("Pool ready:", df_pool.shape)


Pool ready: (724, 25)


In [9]:
import os, json, time
import numpy as np
from openai import OpenAI

client = OpenAI()
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."

def embed_texts(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE, max_retries=6):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def save_npz(path, keys, mat):
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path):
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

# ---- doc embeddings
if DOC_EMBED_NPZ.exists():
    cached_keys, doc_embeds = load_npz(DOC_EMBED_NPZ)
    print("Loaded cached doc embeddings:", doc_embeds.shape)
else:
    cached_keys = df_pool["merge_key"].astype(str).tolist()
    doc_embeds = embed_texts(df_pool["doc_text_trunc"].tolist())
    save_npz(DOC_EMBED_NPZ, cached_keys, doc_embeds)
    print("Computed+saved doc embeddings:", doc_embeds.shape)

key_to_i = {k: i for i, k in enumerate(cached_keys)}

# ---- query embeddings
if QUERY_EMBED_JSON.exists():
    q_cached = json.loads(QUERY_EMBED_JSON.read_text(encoding="utf-8"))
    query_embeds = np.array(q_cached["embeddings"], dtype=np.float32)
    print("Loaded cached query embeddings:", query_embeds.shape)
else:
    query_embeds = embed_texts(query_texts)
    QUERY_EMBED_JSON.write_text(json.dumps({
        "model": EMBED_MODEL,
        "query_texts": query_texts,
        "embeddings": query_embeds.tolist(),
    }, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Computed+saved query embeddings:", query_embeds.shape)


Loaded cached doc embeddings: (724, 1536)
Loaded cached query embeddings: (7, 1536)


In [10]:
def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

docN = l2_normalize(doc_embeds)
qN = l2_normalize(query_embeds)

S = docN @ qN.T
score_embed_max = S.max(axis=1)

df_pool = df_pool.copy()
df_pool["score_embed_max"] = df_pool["merge_key"].astype(str).map(lambda k: score_embed_max[key_to_i[k]])
df_pool["score_embed_norm"] = minmax(df_pool["score_embed_max"].values)

# normalize tfidf within pool for fair fusion
df_pool["score_tfidf_norm"] = minmax(df_pool["score_tfidf"].values)

W_EMBED = 0.60
W_TFIDF = 0.40
df_pool["score_relevance_hybrid"] = W_EMBED * df_pool["score_embed_norm"] + W_TFIDF * df_pool["score_tfidf_norm"]

CITE_WEIGHT = 0.08
df_pool["score_hybrid_pool"] = (1 - CITE_WEIGHT) * df_pool["score_relevance_hybrid"] + CITE_WEIGHT * df_pool["score_cite_norm"]

df_pool.sort_values("score_hybrid_pool", ascending=False).head(10)[
    ["title","score_embed_max","score_tfidf","score_hybrid_pool","score_stageC1"]
]


,title,score_embed_max,score_tfidf,score_hybrid_pool,score_stageC1
216,Personalized pricing and price fairness,0.709698,0.230030,0.865818,0.254376
717,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.613761,0.310356,0.822638,0.285527
708,The Role of Dynamic Pricing to Improve Revenue...,0.629232,0.274905,0.795561,0.252913
568,When should grocery stores adopt time‐based pr...,0.612241,0.280171,0.795082,0.268169
109,Price Discrimination in E-Commerce? An Examina...,0.574440,0.271994,0.786701,0.300494
462,Real-time dynamic Pricing for multiproduct mod...,0.647022,0.233429,0.782006,0.233196
530,DYNAMIC PRICING: THE FUTURE OF RETAIL BUSINESS,0.684874,0.205955,0.781962,0.202616
348,Dynamic pricing with reference price effects i...,0.681583,0.185959,0.773267,0.202993
666,Dynamic Pricing Algorithm Using Reinforcement ...,0.661542,0.224482,0.767503,0.206523
359,Optimal Rebate Strategies Under Dynamic Pricing,0.585023,0.261373,0.765242,0.271340


In [11]:
TOP_FOR_LLM = 150
MAX_ABS_CHARS_FOR_LLM = 1200  # keep costs low

df_llm = (
    df_pool.sort_values("score_hybrid_pool", ascending=False)
    .head(TOP_FOR_LLM)
    .copy()
    .reset_index(drop=True)
)

def llm_doc_text(title, abstract, max_chars=MAX_ABS_CHARS_FOR_LLM):
    t = ("" if pd.isna(title) else str(title)).strip()
    a = ("" if pd.isna(abstract) else str(abstract)).strip().replace("\n", " ")
    a = a[:max_chars]
    return f"TITLE: {t}\nABSTRACT: {a}"

df_llm["llm_text"] = df_llm.apply(lambda r: llm_doc_text(r["title"], r["abstract"]), axis=1)

print("LLM rerank set:", df_llm.shape)
df_llm[["title","score_hybrid_pool"]].head(10)


LLM rerank set: (150, 31)


,title,score_hybrid_pool
0,Personalized pricing and price fairness,0.865818
1,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.822638
2,The Role of Dynamic Pricing to Improve Revenue...,0.795561
3,When should grocery stores adopt time‐based pr...,0.795082
4,Price Discrimination in E-Commerce? An Examina...,0.786701
5,Real-time dynamic Pricing for multiproduct mod...,0.782006
6,DYNAMIC PRICING: THE FUTURE OF RETAIL BUSINESS,0.781962
7,Dynamic pricing with reference price effects i...,0.773267
8,Dynamic Pricing Algorithm Using Reinforcement ...,0.767503
9,Optimal Rebate Strategies Under Dynamic Pricing,0.765242


In [12]:
from pydantic import BaseModel, Field
from agents import Agent, ModelSettings

class RerankScore(BaseModel):
    relevance: int = Field(..., ge=0, le=100, description="Overall relevance to chapter scope (0-100).")
    conceptual: int = Field(..., ge=0, le=100, description="Theory/definitions focus vs implementation (0-100).")
    offline_retail_fit: int = Field(..., ge=0, le=100, description="Brick-and-mortar/offline retail fit (0-100).")
    notes: str = Field(..., description="Very short reason (<= 18 words).")

# Bump this if you change instructions to avoid reusing old cache accidentally
INSTRUCTIONS_VERSION = "v2_scale_0_100"

rerank_agent = Agent(
    name="Chapter Reranker",
    model="gpt-5-nano",
    model_settings=ModelSettings(verbosity="low"),
    instructions=(
        "You are reranking candidate sources for writing a thesis chapter.\n\n"
        "CHAPTER SCOPE:\n"
        "- Definitions + concepts of dynamic pricing in brick-and-mortar (offline/in-store) retail.\n"
        "- Include conceptual approaches: time-dependent pricing, demand-linked price changes.\n"
        "- Useful: fairness/perception, transparency, conceptual framing.\n"
        "- De-emphasize: implementation details (microservices, system architecture, engineering), "
        "algorithm development, RL/ML system design.\n"
        "- De-emphasize: purely online/e-commerce contexts.\n\n"
        "SCORING SCALE (MANDATORY):\n"
        "- Use INTEGER scores from 0 to 100 (NOT 0-10).\n"
        "- 0 = irrelevant\n"
        "- 50 = somewhat relevant\n"
        "- 80 = strong fit\n"
        "- 100 = perfect fit\n"
        "- Use the full range when appropriate.\n\n"
        "OUTPUT FIELDS:\n"
        "- relevance (0-100)\n"
        "- conceptual (0-100)\n"
        "- offline_retail_fit (0-100)\n"
        "- notes: <= 18 words\n\n"
        "Return ONLY the structured output.\n"
    ),
    output_type=RerankScore,
)


In [16]:
import asyncio
import time
import json
import hashlib
from pathlib import Path
import pandas as pd

from agents import Runner
from tqdm.auto import tqdm

# Pricing for gpt-5-nano (per 1M tokens)
PRICE_INPUT_PER_1M  = 0.05
PRICE_CACHED_PER_1M = 0.005
PRICE_OUTPUT_PER_1M = 0.40

# Cache directory (new versioned folder to avoid mixing old outputs)
LLM_CACHE = Path(f".llm_rerank_cache_{INSTRUCTIONS_VERSION}")
LLM_CACHE.mkdir(exist_ok=True)

def cost_from_usage(usage) -> dict:
    """
    Computes token totals + cost from Agents SDK usage.
    Uses per-request entries when available; falls back to aggregated totals.
    """
    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens  = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)

        cost = (
            (non_cached    / 1_000_000) * PRICE_INPUT_PER_1M +
            (cached_tokens / 1_000_000) * PRICE_CACHED_PER_1M +
            (output_tokens / 1_000_000) * PRICE_OUTPUT_PER_1M
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    # Fallback: aggregated totals
    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }

def build_prompt(profile: dict, llm_text: str) -> str:
    return (
        f"CHAPTER MAIN QUERY: {profile['main_query']}\n"
        f"CHAPTER NOTES: {profile.get('notes','')}\n"
        f"FACETS: {', '.join(profile.get('facet_queries', []))}\n\n"
        f"{llm_text}"
    )

def cache_path_for_prompt(prompt: str, model: str, instructions_version: str) -> Path:
    """
    Robust cache key: model + instructions_version + full prompt.
    Prevents reusing stale outputs across chapters/instruction changes.
    """
    blob = model + "\n" + instructions_version + "\n" + prompt
    h = hashlib.sha1(blob.encode("utf-8")).hexdigest()
    return LLM_CACHE / f"{h}.json"

async def rerank_one(idx: int, row, max_retries: int = 6):
    """
    Returns: (idx, scores_dict, usage_dict)
    If loaded from local cache: usage_dict is zeros.
    """
    llm_text = row["llm_text"]
    prompt = build_prompt(profile, llm_text)
    cp = cache_path_for_prompt(prompt, model="gpt-5-nano", instructions_version=INSTRUCTIONS_VERSION)

    # Local cache hit => no API call
    if cp.exists():
        out = json.loads(cp.read_text(encoding="utf-8"))
        usage0 = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
        return idx, out, usage0

    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        try:
            res = await Runner.run(rerank_agent, prompt)
            out = res.final_output.model_dump()
            cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")

            usage = res.context_wrapper.usage
            usage_dict = cost_from_usage(usage)
            return idx, out, usage_dict
        except Exception:
            if attempt == max_retries:
                raise
            await asyncio.sleep(backoff + 0.15 * attempt)
            backoff *= 2

async def rerank_all_concurrent_costed(df_llm: pd.DataFrame, concurrency: int = 100):
    """
    Runs reranking concurrently and returns:
      - df_scores: DataFrame aligned to df_llm order
      - totals: dict with summed tokens + cost for THIS run
    """
    sem = asyncio.Semaphore(concurrency)

    async def wrapped(idx, row):
        async with sem:
            return await rerank_one(idx, row)

    tasks = [asyncio.create_task(wrapped(i, row)) for i, row in df_llm.iterrows()]

    totals = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
    results = {}

    t0 = time.monotonic()
    pbar = tqdm(asyncio.as_completed(tasks), total=len(tasks))

    for fut in pbar:
        idx, out, u = await fut
        results[idx] = out

        totals["requests"] += u["requests"]
        totals["input_tokens"] += u["input_tokens"]
        totals["cached_input_tokens"] += u["cached_input_tokens"]
        totals["output_tokens"] += u["output_tokens"]
        totals["cost_usd"] += u["cost_usd"]

        elapsed = time.monotonic() - t0
        pbar.set_postfix({
            "req": totals["requests"],
            "in_tok": totals["input_tokens"],
            "cached": totals["cached_input_tokens"],
            "out_tok": totals["output_tokens"],
            "cost_$": f"{totals['cost_usd']:.4f}",
            "sec": f"{elapsed:.0f}",
        })

    df_scores = pd.DataFrame([results[i] for i in range(len(df_llm))])
    return df_scores, totals

# ---- run it (10 at a time)
df_scores, totals = await rerank_all_concurrent_costed(df_llm, concurrency=100)

print("\n=== Token + cost summary (this run) ===")
print(totals)

df_scores.head()


  0%|          | 0/150 [00:00<?, ?it/s]


=== Token + cost summary (this run) ===
{'requests': 150, 'input_tokens': 84496, 'cached_input_tokens': 0, 'output_tokens': 195230, 'cost_usd': 0.08231680000000004}


,relevance,conceptual,offline_retail_fit,notes
0,92,90,95,Strong offline pricing concepts; emphasizes fa...
1,70,65,25,Broad dynamic pricing overview; limited offlin...
2,80,85,60,Broad conceptual overview; touches time/demand...
3,90,88,92,Strong theoretical framing of in-store time-ba...
4,25,60,0,Online/e-commerce focus; limited applicability...


In [15]:
mx = df_scores[["relevance","conceptual","offline_retail_fit"]].max().max()
mn = df_scores[["relevance","conceptual","offline_retail_fit"]].min().min()
print("Score range:", mn, "to", mx)
df_scores[["relevance","conceptual","offline_retail_fit","notes"]].head(10)


Score range: 0 to 100


,relevance,conceptual,offline_retail_fit,notes
0,70,65,50,Addresses fairness and personalization; offlin...
1,45,60,25,Online-leaning; useful for pricing concepts an...
2,80,85,60,Broad pricing concepts; mentions time/demand a...
3,90,88,95,Strong theoretical framing; time-based pricing...
4,25,70,15,Online-focused; limited applicability to offli...
5,85,75,85,Time-dependent pricing and demand modeling; st...
6,100,95,100,Strong offline pricing concepts; covers defini...
7,85,90,75,Theoretical intertemporal pricing with referen...
8,25,20,0,Online-focused pricing; limited relevance to o...
9,70,60,85,Rebate-focused; links to time/demand pricing; ...


## C.3

In [17]:
import numpy as np
import pandas as pd

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

# Join rerank scores to the same rows (df_scores is aligned with df_llm order)
df_llm2 = pd.concat([df_llm.reset_index(drop=True), df_scores.reset_index(drop=True)], axis=1)

# Normalize to 0..1
df_llm2["rel_norm"] = df_llm2["relevance"] / 100.0
df_llm2["concept_norm"] = df_llm2["conceptual"] / 100.0
df_llm2["offline_norm"] = df_llm2["offline_retail_fit"] / 100.0

# Primary: LLM scope judgment
df_llm2["score_stageC3"] = (
    0.60 * df_llm2["rel_norm"] +
    0.25 * df_llm2["concept_norm"] +
    0.15 * df_llm2["offline_norm"]
)

# Secondary tie-break: your hybrid semantic score (from Option 3 pool rerank)
df_llm2["hybrid_norm"] = minmax(df_llm2["score_hybrid_pool"].values)

df_llm2["final_score"] = 0.90 * df_llm2["score_stageC3"] + 0.10 * df_llm2["hybrid_norm"]

top50 = (
    df_llm2.sort_values("final_score", ascending=False)
    .head(50)
    .reset_index(drop=True)
)

cols = [
    "final_score","relevance","conceptual","offline_retail_fit",
    "title","year","venue","sources","doi_norm","notes"
]
top50[cols]


,final_score,relevance,conceptual,offline_retail_fit,title,year,venue,sources,doi_norm,notes
0,0.952688,100,95,95,DYNAMIC PRICING: THE FUTURE OF RETAIL BUSINESS,2016.0,NaN,semantic_scholar,NaN,Perfect alignment with offline dynamic pricing...
1,0.927550,92,90,95,Personalized pricing and price fairness,2015.0,International Journal of Industrial Organization,openalex,10.1016/j.ijindorg.2015.11.004,Strong offline pricing concepts; emphasizes fa...
2,0.883474,90,88,92,When should grocery stores adopt time‐based pr...,2023.0,Production and operations management,semantic_scholar,10.1111/poms.14010,Strong theoretical framing of in-store time-ba...
3,0.858588,90,92,80,Dynamic pricing in the presence of reference p...,2020.0,International Journal of Production Research,semantic_scholar,10.1080/00207543.2019.1598592,"Theoretical framework on time-based, demand-li..."
4,0.847385,92,90,95,The multichannel pricing dilemma: Do consumers...,2019.0,International Journal of Research in Marketing,openalex,10.1016/j.ijresmar.2019.01.006,"Strong fit: focuses on definitions, time-based..."
5,0.839070,88,85,92,Optimal pricing and lot-sizing for perishable ...,2013.0,International Journal of Systems Science,semantic_scholar,10.1080/00207721.2011.598956,Strong alignment with time-based pricing of pe...
6,0.823491,85,90,80,Multiunit dynamic pricing with different types...,2024.0,OR spectrum,semantic_scholar,10.1007/s00291-024-00759-x,"Theoretical, multiunit dynamic pricing with ob..."
7,0.823311,90,78,90,Optimal Markdown Pricing and Inventory Allocat...,2017.0,Manufacturing & Service Operations Management,semantic_scholar,10.1287/msom.2016.0609,Markdown pricing with inventory-dependent dema...
8,0.817678,85,90,85,Optimal Dynamic Pricing for Perishable Assets ...,2000.0,NaN,semantic_scholar,10.1287/mnsc.46.3.375.12063,Perishable inventory pricing; time-dependent a...
9,0.815781,88,82,90,Retail pricing models,2023.0,Journal of Revenue and Pricing Management,semantic_scholar,10.1057/s41272-023-00433-x,Strong conceptual fit; ensure explicit in-stor...


In [18]:
# Inspect top 20 titles quickly
top50[["final_score","relevance","offline_retail_fit","title"]].head(20)

,final_score,relevance,offline_retail_fit,title
0,0.952688,100,95,DYNAMIC PRICING: THE FUTURE OF RETAIL BUSINESS
1,0.927550,92,95,Personalized pricing and price fairness
2,0.883474,90,92,When should grocery stores adopt time‐based pr...
3,0.858588,90,80,Dynamic pricing in the presence of reference p...
4,0.847385,92,95,The multichannel pricing dilemma: Do consumers...
5,0.839070,88,92,Optimal pricing and lot-sizing for perishable ...
6,0.823491,85,80,Multiunit dynamic pricing with different types...
7,0.823311,90,90,Optimal Markdown Pricing and Inventory Allocat...
8,0.817678,85,85,Optimal Dynamic Pricing for Perishable Assets ...
9,0.815781,88,90,Retail pricing models


In [19]:
# Look for likely "online" terms in top 50 (we are not filtering yet—just diagnosing)
online_terms = ["online", "e-commerce", "ecommerce", "web", "digital marketplace", "internet"]
t = (top50["title"].fillna("") + " " + top50["notes"].fillna("")).str.lower()

flag_online = t.apply(lambda s: any(w in s for w in online_terms))
print("Top50 flagged as online-ish:", int(flag_online.sum()), "/", len(top50))
top50.loc[flag_online, ["final_score","title","notes"]].head(20)


Top50 flagged as online-ish: 4 / 50


,final_score,title,notes
1,0.927550,Personalized pricing and price fairness,Strong offline pricing concepts; emphasizes fa...
4,0.847385,The multichannel pricing dilemma: Do consumers...,"Strong fit: focuses on definitions, time-based..."
22,0.778649,Dynamic pricing with reference price effects i...,"Dual-channel focus; useful for theory, but onl..."
45,0.719654,How Does Heterogeneous Consumer Behavior Affec...,"Strong theory, includes offline/online contras..."


In [20]:
top50.to_csv("stageC3_top50_reranked.csv", index=False)
print("Saved: stageC3_top50_reranked.csv")


Saved: stageC3_top50_reranked.csv
